In [0]:
%run ./00_config

In [0]:
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_readings"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_readings"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.silver_readings_quarantine"

# tabel silver: tipuri corecte, cheie (lclid, reading_ts), clustering pe household + zi
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
  lclid         STRING    NOT NULL,
  tariff_type   STRING,
  reading_ts    TIMESTAMP NOT NULL,
  reading_date  DATE,
  kwh           DOUBLE,
  _source_file  STRING,
  _ingested_at  TIMESTAMP,
  _silver_ts    TIMESTAMP
) CLUSTER BY (lclid, reading_date)
""")

# carantină: rândurile invalide, păstrate în forma brută + motivul
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {QUARANTINE_TABLE} (
  lclid STRING, stdortou STRING, datetime STRING, kwh_raw STRING,
  reason STRING, _source_file STRING, _ingested_at TIMESTAMP, _quarantined_at TIMESTAMP
)
""")
print(SILVER_TABLE, QUARANTINE_TABLE, sep="\n")

In [0]:
from pyspark.sql import functions as F

# watermark: ultimul _ingested_at deja procesat (minimul dintre silver și carantină)
wm = spark.sql(f"""
  SELECT least(
    coalesce((SELECT max(_ingested_at) FROM {SILVER_TABLE}),     TIMESTAMP'1900-01-01'),
    coalesce((SELECT max(_ingested_at) FROM {QUARANTINE_TABLE}), TIMESTAMP'1900-01-01'))
""").first()[0]
print("procesez rândurile bronze cu _ingested_at >", wm)

parsed = (spark.table(BRONZE_TABLE)
    .filter(F.col("_ingested_at") > F.lit(wm))
    .withColumn("kwh", F.expr("try_cast(trim(kwh_raw) AS DOUBLE)"))          # ' 0.138 ' -> 0.138, 'Null' -> null
    .withColumn("reading_ts", F.expr("try_to_timestamp(substring(datetime, 1, 19), 'yyyy-MM-dd HH:mm:ss')"))
    .withColumn("reason",
        F.when(F.col("lclid").isNull(), "missing_lclid")
         .when(F.col("reading_ts").isNull(), "bad_timestamp")
         .when(F.col("kwh").isNull(), "kwh_not_numeric")
         .when(F.col("kwh") < 0, "negative_kwh")))                       # null = rând valid
display(parsed.groupBy(F.coalesce("reason", F.lit("valid")).alias("status")).count())

In [0]:
# rânduri valide: o singură citire per (lclid, reading_ts), apoi MERGE insert-only
# -> re-rularea nu dublează nimic
good = (parsed.filter(F.col("reason").isNull())
    .select("lclid", F.col("stdortou").alias("tariff_type"), "reading_ts",
            F.to_date("reading_ts").alias("reading_date"), "kwh",
            "_source_file", "_ingested_at", F.current_timestamp().alias("_silver_ts"))
    .dropDuplicates(["lclid", "reading_ts"]))
good.createOrReplaceTempView("silver_updates")

display(spark.sql(f"""
MERGE INTO {SILVER_TABLE} t
USING silver_updates s
ON t.lclid = s.lclid AND t.reading_ts = s.reading_ts
WHEN NOT MATCHED THEN INSERT *
"""))

In [0]:
# rânduri invalide, păstrate brut + motivul; MERGE pe (fișier, household, datetime) -> idempotent
bad = (parsed.filter(F.col("reason").isNotNull())
    .select("lclid", "stdortou", "datetime", "kwh_raw", "reason",
            "_source_file", "_ingested_at", F.current_timestamp().alias("_quarantined_at"))
    .dropDuplicates(["_source_file", "lclid", "datetime"]))
bad.createOrReplaceTempView("quarantine_updates")

display(spark.sql(f"""
MERGE INTO {QUARANTINE_TABLE} t
USING quarantine_updates s
ON t._source_file = s._source_file AND t.lclid <=> s.lclid AND t.datetime <=> s.datetime
WHEN NOT MATCHED THEN INSERT *
"""))

In [0]:
bronze = spark.table(BRONZE_TABLE)
silver = spark.table(SILVER_TABLE)
quar = spark.table(QUARANTINE_TABLE)

# așteptat, calculat independent direct din bronze
valid_bronze = bronze.filter("try_cast(trim(kwh_raw) AS DOUBLE) IS NOT NULL")
expected_silver = valid_bronze.select("lclid", "datetime").distinct().count()

checks = {
    "bronze rows": bronze.count(),
    "silver rows": silver.count(),
    "expected silver rows (distinct valid keys)": expected_silver,
    "quarantine rows": quar.count(),
    "duplicate keys in silver": silver.count() - silver.select("lclid", "reading_ts").distinct().count(),
    "null kwh in silver": silver.filter("kwh IS NULL").count(),
}
for k, v in checks.items():
    print(f"{k:45s} {v:,}")

display(quar.groupBy("reason").count())
display(silver.orderBy("lclid", "reading_ts").limit(10))